# 先导杯：最近全部测试可视化
11次记录，来自5份原始压缩包。诊断与正常测试分开标识，历史不同实例不据此认定加速。

从上到下运行代码格。图可直接查看；数据读取 → 数组计算 → 画图，与课程练习对应。

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
# data.json 与本笔记本放在同一目录
runs = json.loads(Path('data.json').read_text(encoding='utf-8'))['runs']
times = np.array([[v['time_s'] for v in r['result']['loads']] for r in runs])
rates = np.array([[v['consistency_rate'] * 100 for v in r['result']['loads']] for r in runs])
labels = ['10/28 FP32', '10/28 FP16', '10/29 FP32 首轮', '10/29 FP32 补跑', '10/29 cProfile诊断', '10/29 RoPE', '10/29 同晚 FP32', '10/29 编译 第1轮', '10/29 编译 第2轮', '11/01 PR20 新版', '11/01 前版对照']
loads = ['负载1：B1 L32 G256', '负载2：B8 L32 G256', '负载3：B4 L32 G512', '负载4：B4 L512 G64']


## 1. 全部耗时
每个点是一次完整记录的一个负载。横轴从0开始；灰色为cProfile诊断，含诊断开销，不能作正式性能成绩。跨日期仅用于回顾。首轮FP32缺配对日志，其余10轮均与各自日志匹配。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), layout='constrained')
colors = ['#52657c'] * len(runs)
colors[1] = '#bd5952'
colors[4] = '#aaaaaa'
for i in [5, 7, 8]: colors[i] = '#248980'
colors[9] = '#de982b'
for k, ax in enumerate(axes.flat):
    ax.barh(labels, times[:, k], color=colors)
    ax.invert_yaxis()
    ax.set_xlim(0, times[:, k].max()*1.18)
    ax.set_title(loads[k])
    ax.set_xlabel('耗时 / 秒（越小越好）')
    for j, v in enumerate(times[:, k]): ax.text(v, j, f' {v:.3f}', va='center', fontsize=9)
fig.suptitle('全部11轮测试：历史耗时回顾', fontsize=15)
plt.show()


## 2. 输出一致率
颜色与数字表示和公开FP32参考的匹配率，不是模型预测准确率。直接FP16在负载2、3约69%；其他记录四项均100%。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7), layout='constrained')
im = ax.imshow(rates, vmin=60, vmax=100, cmap='YlGnBu', aspect='auto')
ax.set_yticks(np.arange(len(labels)), labels)
ax.set_xticks(np.arange(4), ['负载1', '负载2', '负载3', '负载4'])
for j in range(len(labels)):
    for k in range(4): ax.text(k, j, f'{rates[j,k]:.2f}%', ha='center', va='center', color='white' if rates[j,k]>90 else 'black')
fig.colorbar(im, ax=ax, label='一致率 / %')
ax.set_title('全部11轮：与公开参考输出的一致率')
plt.show()


## 3. 10月29日同晚版本对照
用同晚原版作参照：加速比＝原版耗时÷候选耗时，大于1表示更快。两轮编译结果独立显示，不挑单项最好值；单轮/两轮证据不等于稳定性定论。

In [ ]:
baseline = times[6]
selected = [5, 7, 8]
versions = ['仅RoPE', 'RoPE+编译 第1轮', 'RoPE+编译 第2轮']
x = np.arange(4)
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
for j, (idx, name) in enumerate(zip(selected, versions)):
    values = baseline / times[idx]
    bars = ax.bar(x+(j-1)*0.24, values, width=0.24, label=name)
    ax.bar_label(bars, fmt='%.2f×', padding=3)
ax.axhline(1, color='gray', linestyle='--')
ax.set_xticks(x, ['负载1', '负载2', '负载3', '负载4'])
ax.set_ylim(0, 4.5)
ax.set_ylabel('相对同晚FP32的加速比')
ax.set_title('已有优化的观测收益（不包含编译等待总时长）')
ax.legend()
plt.show()


## 4. 11月1日：PR20是否继续提速
同一实例、各一轮。变化＝（新版耗时÷前版耗时−1）×100%；正数表示新版更慢。这张图专门放大小差异，不能替代第一张图的绝对耗时视图。

In [ ]:
change = (times[9] / times[10] - 1) * 100
fig, ax = plt.subplots(figsize=(9, 4.5), layout='constrained')
bars = ax.bar(['负载1', '负载2', '负载3', '负载4'], change, color='#de982b')
ax.bar_label(bars, labels=[f'+{v:.3f}%' for v in change], padding=4)
ax.axhline(0, color='gray')
ax.set_ylim(-0.4, 0.4)
ax.set_ylabel('新版耗时变化 / %（正数为变慢）')
ax.set_title('PR20本轮未观察到新增收益；差距很小，不能认定稳定变慢')
plt.show()


## 结论与下一步
- 直接FP16：本轮更慢，且两项输出一致率下降。
- RoPE＋编译：已有明显的计时收益，但首次编译等待仍需单独处理。
- PR20：两版四项100%一致；新版本本轮慢约0.24%～0.27%，暂不替换前版。
- 现有数据没有连续请求独立性或编译总成本的完整验证，图中不补造这些结论。

文件来源和精确原始数值保存在 data.json。数据由AI协助提取与核对，DCU测试由用户实际执行。